# Sensing Feature Engineering

Builds weekly sensing features from sensing.csv.
All filtering and aggregation is vectorized.

Pipeline:
  load_sensing (filter to clean UIDs)
    -> quality filter (quality_activity >= 20h)
    -> sensor failure filter (act_still != 86400)
    -> week quality filter (>= 4 valid days per week)
    -> weekly aggregation (mean + std per feature)
    -> save sensing_weekly.csv

Outputs:
  - data/processed/features/sensing_weekly.csv

In [44]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../../")))

from src.data.loaders import load_sensing

DATA_DIR   = Path("../../data/raw/college_experience_dataset")
LABEL_PATH = Path("../../data/processed/labels/weekly_labels.csv")
OUTPUT_DIR = Path("../../data/processed/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Features to aggregate (ep_0 only)
# Audio excluded (Android only)
# sleep_start/end excluded (circular statistics issue)
FEATURES = [
    "sleep_duration",
    "unlock_num_ep_0",
    "unlock_duration_ep_0",
    "act_still_ep_0",
    "act_in_vehicle_ep_0",
    "act_on_bike_ep_0",
    "loc_self_dorm_dur",
    "loc_social_dur",
    "loc_study_dur",
    "other_playing_duration_ep_0",
]

QUALITY_THRESHOLD  = 20    # hours
MIN_QUALITY_DAYS   = 4     # days per week
ACT_STILL_MAX      = 82800 # seconds

---
## Load

Loads sensing data filtered to the clean EMA population.
clean_uids comes from the label table produced in 02_label_engineering.

In [45]:
labels = pd.read_csv(LABEL_PATH)
clean_uids = labels["uid"].unique().tolist()
print(f"Clean UIDs from label table: {len(clean_uids)}")

SENSING_PATH = DATA_DIR / "Sensing" / "sensing.csv"
df = load_sensing(str(SENSING_PATH), clean_uids=clean_uids)

print(f"\nSensing data loaded:")
print(f"  Rows     : {len(df):,}")
print(f"  Students : {df['uid'].nunique()}")
print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"  Columns  : {df.shape[1]}")

Clean UIDs from label table: 216

Sensing data loaded:
  Rows     : 216,007
  Students : 216
  Date range: 2017-09-07 to 2022-06-15
  Columns  : 653


---
## Quality Filter

Removes days where quality_activity < 20h.
Low quality days represent days where the phone was off, dead, or left at home. 18.7% of days are removed by this filter.

In [46]:
n_before = len(df)
df_quality = df[df["quality_activity"] >= QUALITY_THRESHOLD].copy()
n_after = len(df_quality)

print(f"Rows before quality filter : {n_before:,}")
print(f"Rows after quality filter  : {n_after:,}")
print(f"Rows removed               : {n_before - n_after:,} ({(n_before - n_after)/n_before*100:.1f}%)")
print(f"Students remaining         : {df_quality['uid'].nunique()}")

Rows before quality filter : 216,007
Rows after quality filter  : 175,617
Rows removed               : 40,390 (18.7%)
Students remaining         : 215


In [47]:
students_before = set(df["uid"].unique())
students_after  = set(df_quality["uid"].unique())
lost = students_before - students_after

print(f"Student lost at quality filter: {len(lost)}")
for uid in lost:
    rows = df[df["uid"] == uid]
    print(f"  {uid}: {len(rows)} total days, "
          f"max quality_activity = {rows['quality_activity'].max():.1f}h")

Student lost at quality filter: 1
  9ffbe25e279de21de86acd54b6fa60d0: 54 total days, max quality_activity = 19.0h


---
## Sensor Failure Filter

Removes days where act_still_ep_0 = 86,400 seconds (24 hours).
This is a known sensor failure where the accelerometer reported the student as completely still for an entire day.

In [48]:
n_before = len(df_quality)
df_quality = df_quality[df_quality["act_still_ep_0"] < ACT_STILL_MAX].copy()
n_after = len(df_quality)

print(f"Rows before sensor failure filter : {n_before:,}")
print(f"Rows after sensor failure filter  : {n_after:,}")
print(f"Rows removed                      : {n_before - n_after:,}")
print(f"\nact_still_ep_0 max after filter: {df_quality['act_still_ep_0'].max():,.0f}s"
      f"({df_quality['act_still_ep_0'].max()/3600:.1f}h)")

Rows before sensor failure filter : 175,617
Rows after sensor failure filter  : 166,233
Rows removed                      : 9,384

act_still_ep_0 max after filter: 82,799s(23.0h)


---
## Week Quality Filter

Requires at least 4 valid days per (student, week).
Weeks with fewer valid days produce unreliable weekly estimates.
90.3% of student-weeks pass this threshold.

In [49]:
week_quality = (
    df_quality.groupby(["uid", "year_week"])
    .size()
    .reset_index(name="valid_days")
)

print("Valid days per student-week distribution:")
print(week_quality["valid_days"].value_counts().sort_index())

n_weeks_before = len(week_quality)
good_weeks = week_quality[week_quality["valid_days"] >= MIN_QUALITY_DAYS]
n_weeks_after = len(good_weeks)

print(f"\nStudent-weeks before filter : {n_weeks_before:,}")
print(f"Student-weeks after filter  : {n_weeks_after:,}")
print(f"Weeks removed               : {n_weeks_before - n_weeks_after:,} "
      f"({(n_weeks_before - n_weeks_after)/n_weeks_before*100:.1f}%)")

df_final = df_quality.merge(
    good_weeks[["uid", "year_week"]],
    on=["uid", "year_week"],
    how="inner"
)
print(f"\nRows after week quality filter : {len(df_final):,}")
print(f"Students remaining             : {df_final['uid'].nunique()}")

Valid days per student-week distribution:
valid_days
1     1220
2     1051
3     1272
4     1471
5     2335
6     5247
7    15722
Name: count, dtype: int64

Student-weeks before filter : 28,318
Student-weeks after filter  : 24,775
Weeks removed               : 3,543 (12.5%)

Rows after week quality filter : 159,095
Students remaining             : 215


---
## is_ios Platform Feature

is_ios is a student-level property, so it is extracted once per student.

In [50]:
is_ios = (
    df_final.groupby("uid")["is_ios"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={"is_ios": "is_ios"})
)

print(f"Students with is_ios=1 (iOS)    : {(is_ios['is_ios']==1).sum()}")
print(f"Students with is_ios=0 (Android): {(is_ios['is_ios']==0).sum()}")

Students with is_ios=1 (iOS)    : 193
Students with is_ios=0 (Android): 22


---
## Weekly Aggregation

Computes mean and std for each feature across valid days in each
(student, week). Mean captures the typical level. Std captures
consistency.

Location columns will have NaN for some weeks due to missingness.
NaN is left as NaN — zero has a real and distinct value.

In [51]:
existing_features = [f for f in FEATURES if f in df_final.columns]
missing_features = [f for f in FEATURES if f not in df_final.columns]
if missing_features:
    print(f"WARNING: features not found in file: {missing_features}")

# Vectorized groupby aggregation 
agg_dict = {f: ["mean", "std"] for f in existing_features}

weekly = (
    df_final.groupby(["uid", "year_week"])
    .agg(agg_dict)
    .reset_index()
)

# Flatten multi-level column names
weekly.columns = [
    "_".join(col).strip("_") if col[1] else col[0]
    for col in weekly.columns
]

weekly = weekly.merge(is_ios, on="uid", how="left")

print(f"Weekly feature table:")
print(f"  Rows     : {len(weekly):,}")
print(f"  Students : {weekly['uid'].nunique()}")
print(f"  Columns  : {weekly.shape[1]}")
print(f"  Weeks    : {weekly['year_week'].nunique()}")
print(f"  Date range: {weekly['year_week'].min()} to {weekly['year_week'].max()}")
print(f"\nColumns: {list(weekly.columns)}")

Weekly feature table:
  Rows     : 24,775
  Students : 215
  Columns  : 23
  Weeks    : 248
  Date range: 2017-W37 to 2022-W23

Columns: ['uid', 'year_week', 'sleep_duration_mean', 'sleep_duration_std', 'unlock_num_ep_0_mean', 'unlock_num_ep_0_std', 'unlock_duration_ep_0_mean', 'unlock_duration_ep_0_std', 'act_still_ep_0_mean', 'act_still_ep_0_std', 'act_in_vehicle_ep_0_mean', 'act_in_vehicle_ep_0_std', 'act_on_bike_ep_0_mean', 'act_on_bike_ep_0_std', 'loc_self_dorm_dur_mean', 'loc_self_dorm_dur_std', 'loc_social_dur_mean', 'loc_social_dur_std', 'loc_study_dur_mean', 'loc_study_dur_std', 'other_playing_duration_ep_0_mean', 'other_playing_duration_ep_0_std', 'is_ios']


---
## Verification

In [52]:
# Check no act_still values at 86400 survived
still_max = weekly["act_still_ep_0_mean"].max()
if still_max >= ACT_STILL_MAX:
    print(f"WARNING: act_still_mean max is {still_max} - sensor failure rows may remain")
else:
    print(f"Verified: act_still_ep_0_mean max = {still_max:,.0f}s (below {ACT_STILL_MAX})")

# NaN report for location columns
print("\nNaN rates for location features (expected due to GPS missingness):")
loc_cols = [c for c in weekly.columns if "loc_" in c]
for col in loc_cols:
    pct = weekly[col].isna().mean() * 100
    print(f"  {col}: {pct:.1f}% NaN")

# Pre/post COVID split preview
pre  = weekly[weekly["year_week"] < "2020-W12"]
post = weekly[weekly["year_week"] >= "2020-W12"]
print(f"\nPre-COVID student-weeks  : {len(pre):,}")
print(f"Post-COVID student-weeks : {len(post):,}")

Verified: act_still_ep_0_mean max = 82,405s (below 82800)

NaN rates for location features (expected due to GPS missingness):
  loc_self_dorm_dur_mean: 0.1% NaN
  loc_self_dorm_dur_std: 0.2% NaN
  loc_social_dur_mean: 0.1% NaN
  loc_social_dur_std: 0.2% NaN
  loc_study_dur_mean: 0.1% NaN
  loc_study_dur_std: 0.2% NaN

Pre-COVID student-weeks  : 14,172
Post-COVID student-weeks : 10,603


---
## Save

In [53]:
out_path = OUTPUT_DIR / "sensing_weekly.csv"
weekly.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows  : {len(weekly):,}")
print(f"Size  : {out_path.stat().st_size / 1024:.1f} KB")

Saved: ..\..\data\processed\features\sensing_weekly.csv
Rows  : 24,775
Size  : 7729.2 KB
